In [ ]:
import os
import sys
import numpy as np
import pandas as pd

# 1. MONTAR GOOGLE DRIVE Y DEFINIR RUTAS
EN_COLAB = os.path.exists('/content') and ('google.colab' in sys.modules or os.path.exists('/content/drive'))
CARPETA_DRIVE = "Paderborn_QML"
NOMBRE_PIPELINE = "PipeLine_Log1p_clipping_RobustScaling"

drive_root = '/content/drive/MyDrive' if os.path.exists('/content/drive/MyDrive') else '/content/drive/My Drive'
if EN_COLAB and not os.path.exists(drive_root):
    from google.colab import drive
    drive.mount('/content/drive')
    drive_root = '/content/drive/MyDrive' if os.path.exists('/content/drive/MyDrive') else '/content/drive/My Drive'

if os.path.exists(drive_root):
    DRIVE_DIR = os.path.join(drive_root, CARPETA_DRIVE, NOMBRE_PIPELINE)
    os.makedirs(DRIVE_DIR, exist_ok=True)
    DIR_SANOS_DRIVE = os.path.join(DRIVE_DIR, '01_datos', 'sanos')
    os.makedirs(DIR_SANOS_DRIVE, exist_ok=True)
    rutas_candidatas_in = [
        os.path.join(DRIVE_DIR, '01_datos', 'crudos', 'paderborn_features_final.csv'),
        os.path.join(DRIVE_DIR, 'paderborn_features_final.csv'),
        os.path.join(drive_root, CARPETA_DRIVE, '01_datos', 'crudos', 'paderborn_features_final.csv'),
        os.path.join(drive_root, CARPETA_DRIVE, 'paderborn_features_final.csv'),
        '/content/paderborn_features_final.csv',
        'paderborn_features_final.csv',
        '../paderborn_features_final.csv',
        '../../paderborn_features_final.csv',
    ]
    ruta_salida = os.path.join(DRIVE_DIR, 'paderborn_features_tratadas.csv')
    ruta_salida_sanos = os.path.join(DIR_SANOS_DRIVE, 'paderborn_features_tratadas.csv')
else:
    DRIVE_DIR = '.'
    DIR_SANOS_DRIVE = '.'
    rutas_candidatas_in = [
        'paderborn_features_final.csv',
        '../paderborn_features_final.csv',
        '../../paderborn_features_final.csv',
    ]
    ruta_salida = 'paderborn_features_tratadas.csv'
    ruta_salida_sanos = 'paderborn_features_tratadas.csv'

ruta_csv = next((r for r in rutas_candidatas_in if os.path.exists(r)), None)
if ruta_csv is None:
    raise FileNotFoundError("No se encontró paderborn_features_final.csv en Drive ni localmente.")

print(f"📂 Cargando dataset crudo desde: {ruta_csv}")
df = pd.read_csv(ruta_csv)
print(f"✅ Dataset cargado. Dimensiones iniciales: {df.shape}")

# 3. COMPROBACIÓN Y TRATAMIENTO DE NULOS E INFINITOS
df = df.replace([np.inf, -np.inf], np.nan)
n_nulos = df.isnull().sum().sum()
print(f"\n🔍 Verificación de calidad:")
print(f"   • Valores nulos/infinitos detectados: {n_nulos}")

if n_nulos > 0:
    df = df.dropna().reset_index(drop=True)
    print(f"   ✅ Se eliminaron registros corruptos. Nuevas dimensiones: {df.shape}")
else:
    print("   ✅ El dataset está 100% libre de nulos e infinitos.")

# 4. IDENTIFICACIÓN DE COLUMNAS
op_cols = ['RPM', 'Torque_Nm', 'Fuerza_N']
stat_signal_cols = ['RMS', 'Kurtosis', 'Skewness', 'Crest_Factor']
fft_cols = [col for col in df.columns if col.startswith('fft_')]
target_col = [col for col in df.columns if col.lower() in ['label', 'target', 'state', 'fault', 'clase', 'condition']]
col_etiqueta = target_col[0] if target_col else 'Label'
signal_cols = stat_signal_cols + fft_cols

print(f"\n📊 Estructura identificada:")
print(f"   • Espectro FFT: {len(fft_cols)} columnas")
print(f"   • Columna objetivo: '{col_etiqueta}'")

# 5. TRANSFORMACIÓN LOGARÍTMICA EN FFT
df[fft_cols] = np.log1p(np.abs(df[fft_cols]))
print(f"\n✅ Transformación logarítmica log(1+x) aplicada a las {len(fft_cols)} columnas FFT.")

# 6. SEPARACIÓN DE CLASES Y TRATAMIENTO DE ATÍPICOS (CLIPPING)
mask_sano = df[col_etiqueta].astype(str).str.contains('Normal|sano|healthy|0', case=False, regex=True)
df_sano = df[mask_sano].copy()
df_falla = df[~mask_sano].copy()

p_lower = np.percentile(df_sano[signal_cols], 1, axis=0)
p_upper = np.percentile(df_sano[signal_cols], 99, axis=0)
df_sano[signal_cols] = np.clip(df_sano[signal_cols].values, p_lower, p_upper)
print("✅ Clipping (1%-99%) aplicado únicamente a las variables de señal (RMS, estadísticas y FFT).")

# 7. CONSOLIDACIÓN Y EXPORTACIÓN
df_tratado = pd.concat([df_sano, df_falla], axis=0).reset_index(drop=True)
df_tratado.to_csv(ruta_salida, index=False)
if ruta_salida_sanos != ruta_salida:
    df_tratado.to_csv(ruta_salida_sanos, index=False)

print("\n" + "="*70)
print("PROCESAMIENTO COMPLETADO EXITOSAMENTE")
print("="*70)
print(f"• Dimensiones finales: {df_tratado.shape}")
print(f"💾 Archivo guardado:")
print(f"   • {ruta_salida}")
if ruta_salida_sanos != ruta_salida:
    print(f"   • {ruta_salida_sanos}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Dataset cargado. Dimensiones iniciales: (59867, 72)

🔍 Verificación de calidad:
   • Valores nulos/infinitos detectados: 0
   ✅ El dataset está 100% libre de nulos e infinitos.

📊 Estructura identificada:
   • Espectro FFT: 64 columnas
   • Columna objetivo: 'Label'

✅ Transformación logarítmica log(1+x) aplicada a las 64 columnas FFT.
✅ Clipping (1%-99%) aplicado únicamente a las variables de señal (RMS, estadísticas y FFT).

PROCESAMIENTO COMPLETADO EXITOSAMENTE
• Dimensiones finales: (59867, 72)
💾 Archivo guardado en Google Drive:
   /content/drive/My Drive/Paderborn_QML/paderborn_features_tratadas.csv
